# Paper-ready GNNHAR Reproduction

This notebook is the structured Colab entrypoint for reproducing the paper-ready Dow30, S&P100, and S&P500 artifacts. Run the smoke-test cells first. Full-model cells can be expensive and should be launched only after data paths are verified.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, subprocess, json, time, shutil

REPO = Path('/content/GNNHAR')
BRANCH = '2026-06-01'
DRIVE_ROOT = Path('/content/drive/MyDrive/GNNHAR_Research')
RUN_ID = time.strftime('paper-ready-%Y%m%dT%H%M%SZ', time.gmtime())
OUT_ROOT = DRIVE_ROOT / 'runs' / RUN_ID
PAPER_READY = DRIVE_ROOT / 'results' / 'paper_ready_20260617'
OUT_ROOT.mkdir(parents=True, exist_ok=True)
print('RUN_ID =', RUN_ID)
print('OUT_ROOT =', OUT_ROOT)
print('PAPER_READY =', PAPER_READY)


In [ ]:
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, 'https://github.com/easygl1der/GNNHAR.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', BRANCH], check=True)
os.chdir(REPO)
subprocess.run(['git', 'log', '--oneline', '-3'], check=True)
subprocess.run(['pip', 'install', '-q', '-r', 'requirements-scale.txt'], check=True)


## Verify Data and GPU

This cell does not train. It verifies that the expected data folders exist and records the GPU name.


In [ ]:
for path in [
    Path('experiments/dow30/data'),
    Path('data/scale_experiment/sp100'),
    Path('data/scale_experiment/sp500'),
]:
    print(path, 'exists=', path.exists())

try:
    subprocess.run(['nvidia-smi'], check=False)
except FileNotFoundError:
    print('nvidia-smi not found; CPU runtime or non-GPU Colab session')

subprocess.run(['python', 'scripts/analysis/audit_scale_experiment_data.py', '--output-dir', str(OUT_ROOT / 'audit')], check=True)
print((OUT_ROOT / 'audit' / 'scale_data_method_audit.md').read_text()[:3000])


## Smoke Test

This checks the rolling pipeline on a small subset. Do not use smoke-test losses in the paper.


In [ ]:
smoke_cmd = [
    'python', 'scripts/analysis/run_zhang_scale_colab_full.py',
    '--output-root', str(OUT_ROOT / 'smoke'),
    '--universes', 'dow30,sp100',
    '--losses', 'MSE',
    '--horizons', '1',
    '--models', 'HAR,GHAR,HAR+IV,GHAR+IV,GNNHAR1L,GNNHAR1L-IV',
    '--hidden-grid', '9',
    '--lr-grid', '0.001',
    '--epochs', '20',
    '--mcs-bootstrap', '20',
    '--max-blocks', '1',
    '--max-tickers', '20',
    '--allow-missing-gpu',
]
subprocess.run(smoke_cmd, check=True)


## Full Dow30 and S&P100 Colab Runs

Run this cell when the smoke test passes. It exports full-model rolling outputs. The Dow30 output should later be copied into `paper_ready_20260617/universes/dow30/aligned_full_model_20260619/`.


In [ ]:
full_cmd = [
    'python', 'scripts/analysis/run_zhang_scale_colab_full.py',
    '--output-root', str(OUT_ROOT / 'full_colab'),
    '--universes', 'dow30,sp100',
    '--losses', 'MSE,QLIKE',
    '--horizons', '1',
    '--models', 'HAR,GHAR,HAR+IV,GHAR+IV,GNNHAR1L,GNNHAR2L,GNNHAR3L,GNNHAR4L,GNNHAR5L,GNNHAR1L-IV,GNNHAR2L-IV,GNNHAR3L-IV,GNNHAR4L-IV,GNNHAR5L-IV',
    '--hidden-grid', '9',
    '--lr-grid', '0.001',
    '--epochs', '5000',
    '--batch-size', '128',
    '--num-nn', '1',
    '--mcs-bootstrap', '10000',
    '--lookback', '1000',
    '--window', '22',
    '--valid-len', '22',
    '--block-stride', '22',
]
print('Launching full Colab run:')
print(' '.join(full_cmd))
# Uncomment to run.
# subprocess.run(full_cmd, check=True)


## S&P500 AutoDL Import

S&P500 can be run on AutoDL and copied into Drive. Keep the 449-ticker AutoDL output as the formal S&P500 source. Do not replace it with older smaller Google Drive uploads.


In [ ]:
SP500_AUTODL_ARCHIVE = DRIVE_ROOT / 'autodl' / 'sp500_results_sp500_parallel_a100_20260617T0706Z.tar.gz'
print('Expected S&P500 AutoDL archive:', SP500_AUTODL_ARCHIVE)
print('exists=', SP500_AUTODL_ARCHIVE.exists())
# Use scripts/analysis/convert_zhang_scale_to_colab_outputs.py after extracting the archive if needed.


## Export Paper-ready Diagnostics

After aligned arrays are present under the paper-ready layout, recompute MCS and diagnostics. This is the step that produces the Dow30 234-date MCS needed by the manuscript.


In [ ]:
DOW30_RUN = PAPER_READY / 'universes' / 'dow30' / 'aligned_full_model_20260619'
cmd = [
    'python', 'scripts/paper_ready/recompute_diagnostics.py',
    '--run-dir', str(DOW30_RUN),
    '--bootstrap', '10000',
    '--block-size', '2',
    '--algorithm', 'SQ',
    '--seed', '0',
]
print('Run after DOW30_RUN contains arrays/truth.npy and predictions/pred_*.npy:')
print(' '.join(cmd))
# subprocess.run(cmd, check=True)


## Build Manuscript PDF

After diagnostics and LaTeX tables are refreshed, compile the paper.


In [ ]:
paper_dir = REPO / 'reports' / 'zhang_style_statistics_20260618' / 'paper_draft'
subprocess.run(['latexmk', '-pdf', '-interaction=nonstopmode', 'main.tex'], cwd=paper_dir, check=True)
print(paper_dir / 'main.pdf')
